# Open Hair Stack — follicular-unit training pipeline (one click)

Everything needed to produce `fu-v0.1`: install → download the CC-BY dataset → convert → train → evaluate → publish weights.

**What you need:** a Google account and (for the dataset step) a free Roboflow API key: https://docs.roboflow.com/developer/api/authentication
**What you get:** `best.pt` + eval metrics + a Hugging Face model repo — all published. Runtime: ~90 minutes on a free T4.

Run all cells top to bottom. Do not skip the dataset step; a trained model with no data provenance is not publishable under our Charter.

## Step 1 — Install
Restarts the runtime implicitly once on first run; if the import below fails, run this cell again.

In [ ]:
!pip install -q ultralytics==8.3.0 huggingface_hub==0.26.0
import ultralytics
print('ultralytics', ultralytics.__version__)

## Step 2 — Get the dataset
Source: **ranaoudio / hair-follicle-detection-gjgak** — 688 images, hair-count-per-unit classes, CC BY 4.0.
Recorded in `openhairstack/data-pipeline` → `DATASETS.md` with the required attribution.

Option A (recommended): paste your free Roboflow key below.
Option B: download the zip from the dataset page, unzip, set `EXPORT_DIR` to the folder containing `data.yaml` + `train/valid/test`.

In [ ]:
ROBOFLOW_API_KEY = ''  # paste key here (starts with rf_)
workspace, project, version = 'ranaudio', 'hair-follicle-detection-gjgak', 1

from roboflow import Roboflow
import shutil

if ROBOFLOW_API_KEY:
    ds = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(workspace).project(project).version(version).download('yolov8')
    EXPORT_DIR = ds.location
else:
    EXPORT_DIR = '/content/dataset'  # Option B: put the unzipped export here (must contain data.yaml + train/valid/test)

print('EXPORT_DIR =', EXPORT_DIR)

## Step 3 — Convert to the canonical release layout
Moves the export into `images/{train,val,test}` + `labels/...` + `dataset.yaml`, remaps source class ids `1..5` to our taxonomy `fu1..fu5` (hair count per follicular unit).

In [ ]:
from pathlib import Path
from collections import Counter

SRC = Path(EXPORT_DIR)
OUT = Path('/content/follicle-obb-v0.1')
CANON = ['fu1', 'fu2', 'fu3', 'fu4', 'fu5']
SPLIT_MAP = {'train': 'train', 'val': 'valid', 'test': 'test'}
CLASS_MAP = {'1': 'fu1', '2': 'fu2', '3': 'fu3', '4': 'fu4', '5': 'fu5'}
PREFIX = 'ranaudio_'  # source tag keeps filenames unique when we merge more datasets

counts = Counter()
for our_split, src_split in SPLIT_MAP.items():
    img_root, lbl_root = SRC / src_split / 'images', SRC / src_split / 'labels'
    (OUT / 'images' / our_split).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / our_split).mkdir(parents=True, exist_ok=True)
    if not img_root.exists():
        print(f'  {src_split}: absent, skipped'); continue
    for img in sorted(img_root.rglob('*')):
        if img.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.bmp', '.webp'):
            continue
        lbl = lbl_root / f'{img.stem}.txt'
        if not lbl.exists():
            print(f'  {img.name}: no label, skipped'); continue
        lines = []
        for line in lbl.read_text().splitlines():
            p = line.split()
            if len(p) >= 5 and p[0] in CLASS_MAP:
                lines.append(' '.join([str(CANON.index(CLASS_MAP[p[0]]))] + p[1:]))
                counts[CLASS_MAP[p[0]]] += 1
        if not lines:
            continue
        stem = PREFIX + img.stem
        shutil.copy2(img, OUT / 'images' / our_split / f'{stem}{img.suffix.lower()}')
        (OUT / 'labels' / our_split / f'{stem}.txt').write_text('\n'.join(lines) + '\n')

(OUT / 'dataset.yaml').write_text(
    f'path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n' +
    ''.join(f'  {i}: {c}\n' for i, c in enumerate(CANON)))
print('instances per class:', dict(counts))
assert sum(counts.values()) > 500, 'conversion failed - fix before training'

## Step 4 — Train (`fu-v0.1`)
yolo11n, 60 epochs, 640px — ~20–60 min on a free T4. Patience 25 for early stop.
Run it once, note the metrics, then iterate (epochs/imgsz/aug) with run names `fu-v0.1-*`.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data=str(OUT / 'dataset.yaml'),
    epochs=60,
    imgsz=640,
    batch=16,
    name='fu-v0.1',
    project='/content/runs',
    seed=0,
    patience=25,
    device=0,
    exist_ok=True,
)

In [ ]:
import json, datetime
from pathlib import Path

run_dir = Path('/content/runs/fu-v0.1')
provenance = {
    'command': 'yolo detect train data=follicle-obb-v0.1 model=yolo11n.pt epochs=60 imgsz=640 batch=16 seed=0',
    'dataset': 'ranaudio/hair-follicle-detection-gjgak (688 imgs, CC BY 4.0), Roboflow Universe, 2025',
    'trained_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'platform': 'google colab GPU runtime (note the GPU type in the run log)'
}
(run_dir / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print('provenance written')

## Step 5 — Evaluate on the val split
Copy these numbers into the model card and the README benchmark table. If `fu3`/`fu4` recall is poor, say so in the card — honesty is the project's marketing strategy.

In [ ]:
best = YOLO('/content/runs/fu-v0.1/weights/best.pt')
metrics = best.val(data=str(OUT / 'dataset.yaml'), split='val', imgsz=640)

print(f'mAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')
print(f'precision : {metrics.box.mp:.3f}')
print(f'recall    : {metrics.box.mr:.3f}')
print('\nper class:')
for i, name in metrics.names.items():
    print(f'  {name:4s} mAP50={metrics.box.maps[i]:.3f}  P={metrics.box.p[i]:.3f}  R={metrics.box.r[i]:.3f}')

## Step 6 — Sanity preview
One inference on a val image; the annotated preview is also what goes in the launch tweet.

In [ ]:
from PIL import Image
import glob

samples = sorted(glob.glob(str(OUT / 'images' / 'val' / '*.*')))
print('val images:', len(samples))
res = best(samples[0], conf=0.25)
annotated = res[0].plot()
Image.fromarray(annotated).save('/content/preview.jpg')
from IPython.display import Image as IPImage
IPImage(filename='/content/preview.jpg')

## Step 7 — Publish weights to Hugging Face
Needs a **write** token: https://huggingface.co/settings/tokens — leave blank to skip (the model card template is `follicle-detect/models/MODEL_CARD.md`).
The repo defaults to **your own namespace** so this works before the Open Hair Stack org exists: `<you>/follicle-fu-v0.1`.

In [ ]:
HF_TOKEN = ''  # paste write token, leave empty to skip publishing

from huggingface_hub import HfApi
from pathlib import Path

if HF_TOKEN:
    api = HfApi()
    user = api.whoami(token=HF_TOKEN)['name']
    REPO = f'{user}/follicle-fu-v0.1'
    api.create_repo(REPO, repo_type='model', exist_ok=True)
    for f in ['/content/runs/fu-v0.1/weights/best.pt',
              '/content/runs/fu-v0.1/provenance.json']:
        api.upload_file(path_or_fileobj=f, path_in_repo=Path(f).name,
                        repo_id=REPO, token=HF_TOKEN)
        print('uploaded ->', REPO, Path(f).name)
    print('model card to write next: https://huggingface.co/' + REPO)
else:
    print('HF_TOKEN empty - skipped. Set it and rerun this cell.')

In [ ]:
# backup: keep the run even if the runtime recycles
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copytree('/content/runs', '/content/drive/MyDrive/openhairstack-runs', dirs_exist_ok=True)
print('backed up to Google Drive: MyDrive/openhairstack-runs')

## Done — what to do next
1. Write the model card: copy `openhairstack/follicle-detect/models/MODEL_CARD.md` into the HF README with today's numbers.
2. Update the README benchmark table in `follicle-detect` with the same numbers.
3. Push the dataset conversion as `follicle-obb-v0.1` (or upload to HF as a dataset).
4. The demo Space auto-loads `<namespace>/follicle-fu-v0.1` once you point `MODEL_ID` at it.
5. Then post the launch thread — with a table of real numbers, not vibes.